# Notebook 05: Intervention Aware Synthetic Progression

## Objective 3

This notebook generates conservative, moderate, and strong synthetic progression scenarios only for students with assigned interventions.

The intervention type controls which mutable features may change. Historical and contextual features remain unchanged. All generated profiles are validated before they are saved for the reassessment module.

The synthetic profiles are scenario based model inputs. They are not evidence that an intervention caused a real student outcome.


## 1. Import libraries


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 200)

SCENARIOS = ["conservative", "moderate", "strong"]
RANDOM_STATE = 42

print("Libraries imported successfully.")


Libraries imported successfully.


## 2. Load intervention cases and raw model features


In [2]:
CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

ML_READY_DIR = PROJECT_DIR / "data" / "processed" 
INTERVENTION_DIR = PROJECT_DIR / "results" / "explainability_interventions"
OUTPUT_DIR = PROJECT_DIR / "results" / "synthetic_progression"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

X_train_raw = pd.read_pickle(ML_READY_DIR / "X_train_raw.pkl")
X_test_raw = pd.read_pickle(ML_READY_DIR / "X_test_raw.pkl")

case_summary = pd.read_csv(
    INTERVENTION_DIR / "high_risk_case_summary.csv"
)

intervention_actions = pd.read_csv(
    INTERVENTION_DIR / "student_intervention_actions.csv"
)

case_summary["test_position"] = pd.to_numeric(
    case_summary["test_position"], errors="raise"
).astype(int)

print("Raw training shape:", X_train_raw.shape)
print("Raw testing shape:", X_test_raw.shape)
print("Intervention cases:", case_summary["case_id"].nunique())
print("Intervention action rows:", len(intervention_actions))


Raw training shape: (3008, 23)
Raw testing shape: (752, 23)
Intervention cases: 20
Intervention action rows: 82


## 3. Create the intervention log


In [3]:
required_case_columns = {
    "case_id",
    "test_position",
    "predicted_risk_probability",
}

required_action_columns = {
    "case_id",
    "support_area",
    "trigger_features",
    "recommended_action",
}

missing_case_columns = required_case_columns - set(case_summary.columns)
missing_action_columns = required_action_columns - set(intervention_actions.columns)

if missing_case_columns:
    raise KeyError(f"Missing case columns: {sorted(missing_case_columns)}")

if missing_action_columns:
    raise KeyError(f"Missing action columns: {sorted(missing_action_columns)}")

intervention_log = (
    intervention_actions[
        ["case_id", "support_area", "trigger_features", "recommended_action"]
    ]
    .drop_duplicates()
    .copy()
)

intervention_log["intervention_given"] = 1
intervention_log["intervention_status"] = "Planned"

intervention_log = intervention_log.merge(
    case_summary[
        ["case_id", "test_position", "predicted_risk_probability"]
    ],
    on="case_id",
    how="left",
    validate="many_to_one",
)

display(intervention_log.head(20))


,case_id,support_area,trigger_features,recommended_action,intervention_given,intervention_status,test_position,predicted_risk_probability
0,TEST_CASE_0085,Attendance support,"absence_count, attendance_rate","Review attendance barriers, agree an attendance improvement plan, and monitor attendance weekly.",1,Planned,84,1.000000
1,TEST_CASE_0085,Moodle engagement support,"active_days_rate, largest_inactivity_days, total_course_event_clicks","Provide an engagement plan covering course access, learning materials, assessment activity, and regular Moodle participation.",1,Planned,84,1.000000
2,TEST_CASE_0085,Workload adjustment,registered_course_count,Review the student's registered workload and repeated courses with the academic adviser.,1,Planned,84,1.000000
3,TEST_CASE_0085,Referral and coordinated review,High predicted risk or multiple risk domains,Refer the case for coordinated review by the academic adviser and the relevant student support service.,1,Planned,84,1.000000
4,TEST_CASE_0724,Attendance support,"absence_count, attendance_rate","Review attendance barriers, agree an attendance improvement plan, and monitor attendance weekly.",1,Planned,723,0.996667
5,TEST_CASE_0724,Moodle engagement support,"active_days_rate, largest_inactivity_days, zero_activity_days","Provide an engagement plan covering course access, learning materials, assessment activity, and regular Moodle participation.",1,Planned,723,0.996667
6,TEST_CASE_0724,Workload adjustment,registered_course_count,Review the student's registered workload and repeated courses with the academic adviser.,1,Planned,723,0.996667
7,TEST_CASE_0724,Referral and coordinated review,High predicted risk or multiple risk domains,Refer the case for coordinated review by the academic adviser and the relevant student support service.,1,Planned,723,0.996667
8,TEST_CASE_0561,Tutoring and academic support,"previous_cgpa_ordinal, previous_failed_course_count, repeated_course_count_current","Arrange targeted tutoring, assessment support, and an academic progress review.",1,Planned,560,0.990000
9,TEST_CASE_0561,Workload adjustment,"registered_course_count, registered_credits, repeated_course_count_current",Review the student's registered workload and repeated courses with the academic adviser.,1,Planned,560,0.990000


## 4. Define mutable and immutable features


In [4]:
MUTABLE_FEATURES = {
    "attendance_rate",
    "absence_count",
    "accessed_course_count",
    "course_access_rate",
    "total_course_event_clicks",
    "active_days_numeric",
    "active_days_rate",
    "zero_activity_days",
    "largest_inactivity_days",
    "learning_material_events_ordinal",
    "assessment_interaction_events_ordinal",
    "registered_course_count",
    "registered_credits",
    "enrolled_course_count",
}

IMMUTABLE_FEATURES = {
    "programme_or_school",
    "year_level",
    "repeated_course_count_current",
    "previous_term_gpa_ordinal",
    "previous_cgpa_ordinal",
    "previous_failed_course_count",
    "previous_withdrawn_course_count",
    "previous_repeated_course_count",
    "previous_academic_standing",
}

missing_mutable = MUTABLE_FEATURES - set(X_test_raw.columns)
if missing_mutable:
    raise KeyError(f"Missing mutable fields: {sorted(missing_mutable)}")

print("Mutable fields:", len(MUTABLE_FEATURES))
print("Immutable fields found:", len(IMMUTABLE_FEATURES & set(X_test_raw.columns)))


Mutable fields: 14
Immutable fields found: 9


## 5. Derive data constraints


In [5]:
observation_days_values = (
    pd.to_numeric(X_train_raw["active_days_numeric"], errors="coerce")
    + pd.to_numeric(X_train_raw["zero_activity_days"], errors="coerce")
)

OBSERVATION_DAYS = int(round(observation_days_values.dropna().median()))

CLICK_CAP = float(
    pd.to_numeric(
        X_train_raw["total_course_event_clicks"], errors="coerce"
    ).quantile(0.99)
)

MIN_REGISTERED_CREDITS = float(
    pd.to_numeric(
        X_train_raw["registered_credits"], errors="coerce"
    ).dropna().min()
)

print("Observation days:", OBSERVATION_DAYS)
print("Click cap:", round(CLICK_CAP, 2))
print("Minimum registered credits:", MIN_REGISTERED_CREDITS)


Observation days: 34
Click cap: 1615.67
Minimum registered credits: 15.0


## 6. Agreed synthetic scenario rules


In [6]:
RULE_TABLE = pd.DataFrame([
    ["Attendance support", "attendance_rate", "+5 points", "+10 points", "+15 points", "Maximum 100"],
    ["Attendance support", "absence_count", "Reduce 10%", "Reduce 20%", "Reduce 30%", "Minimum 0"],
    ["Moodle engagement support", "accessed_course_count", "+1", "+1", "+2", "Cannot exceed enrolled courses"],
    ["Moodle engagement support", "course_access_rate", "Recalculate", "Recalculate", "Recalculate", "0 to 1"],
    ["Moodle engagement support", "total_course_event_clicks", "+10%", "+20%", "+30%", "Cap at training percentile 99"],
    ["Moodle engagement support", "active_days_numeric", "+2", "+4", "+6", f"Maximum {OBSERVATION_DAYS}"],
    ["Moodle engagement support", "active_days_rate", "Recalculate", "Recalculate", "Recalculate", "0 to 1"],
    ["Moodle engagement support", "zero_activity_days", "Recalculate", "Recalculate", "Recalculate", f"0 to {OBSERVATION_DAYS}"],
    ["Moodle engagement support", "largest_inactivity_days", "-1", "-3", "-5", "Minimum 0"],
    ["Moodle engagement support", "learning_material_events_ordinal", "+1", "+1", "+2", "Maximum 4"],
    ["Tutoring and academic support", "assessment_interaction_events_ordinal", "+1", "+1", "+2", "Maximum 4"],
    ["Tutoring and academic support", "learning_material_events_ordinal", "+1", "+1", "+2", "Maximum 4"],
    ["Tutoring and academic support", "total_course_event_clicks", "+5%", "+10%", "+15%", "Cap at training percentile 99"],
    ["Tutoring and academic support", "active_days_numeric", "+1", "+2", "+3", f"Maximum {OBSERVATION_DAYS}"],
    ["Workload adjustment", "registered_course_count", "No change", "-1", "-2", "Minimum 1"],
    ["Workload adjustment", "registered_credits", "No change", "Reduce proportionally", "Reduce proportionally", "Minimum valid load"],
    ["Workload adjustment", "enrolled_course_count", "No change", "-1", "-2", "Minimum 1"],
    ["Academic counselling", "No direct feature", "No direct change", "No direct change", "No direct change", "Requires linked measurable support"],
    ["Referral and coordinated review", "No direct feature", "No direct change", "No direct change", "No direct change", "Requires linked measurable support"],
], columns=[
    "intervention_type",
    "feature",
    "conservative",
    "moderate",
    "strong",
    "constraint",
])

display(RULE_TABLE)

RULE_TABLE.to_csv(
    OUTPUT_DIR / "synthetic_progression_rule_table.csv",
    index=False,
)


,intervention_type,feature,conservative,moderate,strong,constraint
0,Attendance support,attendance_rate,+5 points,+10 points,+15 points,Maximum 100
1,Attendance support,absence_count,Reduce 10%,Reduce 20%,Reduce 30%,Minimum 0
2,Moodle engagement support,accessed_course_count,+1,+1,+2,Cannot exceed enrolled courses
3,Moodle engagement support,course_access_rate,Recalculate,Recalculate,Recalculate,0 to 1
4,Moodle engagement support,total_course_event_clicks,+10%,+20%,+30%,Cap at training percentile 99
5,Moodle engagement support,active_days_numeric,+2,+4,+6,Maximum 34
6,Moodle engagement support,active_days_rate,Recalculate,Recalculate,Recalculate,0 to 1
7,Moodle engagement support,zero_activity_days,Recalculate,Recalculate,Recalculate,0 to 34
8,Moodle engagement support,largest_inactivity_days,-1,-3,-5,Minimum 0
9,Moodle engagement support,learning_material_events_ordinal,+1,+1,+2,Maximum 4


## 7. Scenario parameters


In [7]:
SCENARIO_PARAMETERS = {
    "conservative": {
        "attendance_points": 5,
        "absence_reduction": 0.10,
        "accessed_courses_add": 1,
        "engagement_click_increase": 0.10,
        "engagement_active_days_add": 2,
        "inactivity_reduction": 1,
        "engagement_ordinal_add": 1,
        "tutoring_click_increase": 0.05,
        "tutoring_active_days_add": 1,
        "tutoring_ordinal_add": 1,
        "workload_course_reduction": 0,
    },
    "moderate": {
        "attendance_points": 10,
        "absence_reduction": 0.20,
        "accessed_courses_add": 1,
        "engagement_click_increase": 0.20,
        "engagement_active_days_add": 4,
        "inactivity_reduction": 3,
        "engagement_ordinal_add": 1,
        "tutoring_click_increase": 0.10,
        "tutoring_active_days_add": 2,
        "tutoring_ordinal_add": 1,
        "workload_course_reduction": 1,
    },
    "strong": {
        "attendance_points": 15,
        "absence_reduction": 0.30,
        "accessed_courses_add": 2,
        "engagement_click_increase": 0.30,
        "engagement_active_days_add": 6,
        "inactivity_reduction": 5,
        "engagement_ordinal_add": 2,
        "tutoring_click_increase": 0.15,
        "tutoring_active_days_add": 3,
        "tutoring_ordinal_add": 2,
        "workload_course_reduction": 2,
    },
}

display(pd.DataFrame(SCENARIO_PARAMETERS).T)


,attendance_points,absence_reduction,accessed_courses_add,engagement_click_increase,engagement_active_days_add,inactivity_reduction,engagement_ordinal_add,tutoring_click_increase,tutoring_active_days_add,tutoring_ordinal_add,workload_course_reduction
conservative,5.0,0.1,1.0,0.1,2.0,1.0,1.0,0.05,1.0,1.0,0.0
moderate,10.0,0.2,1.0,0.2,4.0,3.0,1.0,0.10,2.0,1.0,1.0
strong,15.0,0.3,2.0,0.3,6.0,5.0,2.0,0.15,3.0,2.0,2.0


## 8. Create progression functions


In [8]:
def interventions_for_case(case_id):
    return set(
        intervention_log.loc[
            intervention_log["case_id"] == case_id,
            "support_area",
        ].dropna()
    )


def recalculate_related_features(profile):
    enrolled = max(1, int(round(profile["enrolled_course_count"])))
    accessed = int(round(profile["accessed_course_count"]))
    accessed = max(0, min(accessed, enrolled))

    profile["enrolled_course_count"] = enrolled
    profile["accessed_course_count"] = accessed
    profile["course_access_rate"] = accessed / enrolled

    active_days = int(round(profile["active_days_numeric"]))
    active_days = max(0, min(active_days, OBSERVATION_DAYS))

    profile["active_days_numeric"] = active_days
    profile["active_days_rate"] = active_days / OBSERVATION_DAYS
    profile["zero_activity_days"] = OBSERVATION_DAYS - active_days

    profile["largest_inactivity_days"] = int(
        max(
            0,
            min(
                round(profile["largest_inactivity_days"]),
                profile["zero_activity_days"],
            ),
        )
    )

    return profile


def generate_synthetic_profile(baseline, interventions, scenario):
    p = SCENARIO_PARAMETERS[scenario]
    updated = baseline.copy()
    changed_by = {}

    def mark(feature, intervention):
        changed_by.setdefault(feature, set()).add(intervention)

    if "Attendance support" in interventions:
        updated["attendance_rate"] = min(
            100,
            float(baseline["attendance_rate"]) + p["attendance_points"],
        )
        updated["absence_count"] = max(
            0,
            int(round(float(baseline["absence_count"]) * (1 - p["absence_reduction"])))
        )
        mark("attendance_rate", "Attendance support")
        mark("absence_count", "Attendance support")

    if "Moodle engagement support" in interventions:
        updated["accessed_course_count"] = (
            float(baseline["accessed_course_count"]) + p["accessed_courses_add"]
        )
        updated["total_course_event_clicks"] = min(
            CLICK_CAP,
            float(baseline["total_course_event_clicks"])
            * (1 + p["engagement_click_increase"]),
        )
        updated["active_days_numeric"] = (
            float(baseline["active_days_numeric"])
            + p["engagement_active_days_add"]
        )
        updated["largest_inactivity_days"] = max(
            0,
            float(baseline["largest_inactivity_days"])
            - p["inactivity_reduction"],
        )
        updated["learning_material_events_ordinal"] = min(
            4,
            float(baseline["learning_material_events_ordinal"])
            + p["engagement_ordinal_add"],
        )
        for feature in [
            "accessed_course_count",
            "total_course_event_clicks",
            "active_days_numeric",
            "largest_inactivity_days",
            "learning_material_events_ordinal",
        ]:
            mark(feature, "Moodle engagement support")

    if "Tutoring and academic support" in interventions:
        tutoring_clicks = min(
            CLICK_CAP,
            float(baseline["total_course_event_clicks"])
            * (1 + p["tutoring_click_increase"]),
        )
        updated["total_course_event_clicks"] = max(
            float(updated["total_course_event_clicks"]),
            tutoring_clicks,
        )
        updated["active_days_numeric"] = max(
            float(updated["active_days_numeric"]),
            float(baseline["active_days_numeric"])
            + p["tutoring_active_days_add"],
        )
        for feature in [
            "learning_material_events_ordinal",
            "assessment_interaction_events_ordinal",
        ]:
            updated[feature] = min(
                4,
                max(
                    float(updated[feature]),
                    float(baseline[feature]) + p["tutoring_ordinal_add"],
                ),
            )
            mark(feature, "Tutoring and academic support")
        mark("total_course_event_clicks", "Tutoring and academic support")
        mark("active_days_numeric", "Tutoring and academic support")

    if "Workload adjustment" in interventions:
        reduction = int(p["workload_course_reduction"])
        if reduction > 0:
            original_courses = max(1, int(round(baseline["registered_course_count"])))
            new_courses = max(1, original_courses - reduction)
            credits_per_course = float(baseline["registered_credits"]) / original_courses

            updated["registered_course_count"] = new_courses
            updated["registered_credits"] = max(
                MIN_REGISTERED_CREDITS,
                round(new_courses * credits_per_course),
            )
            updated["enrolled_course_count"] = max(
                1,
                int(round(baseline["enrolled_course_count"])) - reduction,
            )
            mark("registered_course_count", "Workload adjustment")
            mark("registered_credits", "Workload adjustment")
            mark("enrolled_course_count", "Workload adjustment")

    updated = recalculate_related_features(updated)
    return updated, changed_by


## 9. Generate conservative, moderate, and strong profiles


In [9]:
profile_rows = []
change_rows = []

for _, case in case_summary.iterrows():
    case_id = case["case_id"]
    position = int(case["test_position"])
    baseline = X_test_raw.iloc[position].copy()
    interventions = interventions_for_case(case_id)

    for scenario in SCENARIOS:
        if interventions:
            updated, changed_by = generate_synthetic_profile(
                baseline,
                interventions,
                scenario,
            )
        else:
            updated = baseline.copy()
            changed_by = {}

        profile_row = {
            "case_id": case_id,
            "test_position": position,
            "scenario": scenario,
            "intervention_given": int(bool(interventions)),
            "intervention_types": " | ".join(sorted(interventions)),
            "baseline_risk_probability": case["predicted_risk_probability"],
        }
        profile_row.update(updated.to_dict())
        profile_rows.append(profile_row)

        for feature in X_test_raw.columns:
            old_value = baseline[feature]
            new_value = updated[feature]

            same = (
                (pd.isna(old_value) and pd.isna(new_value))
                or old_value == new_value
            )

            if not same:
                change_rows.append({
                    "case_id": case_id,
                    "test_position": position,
                    "scenario": scenario,
                    "feature": feature,
                    "baseline_value": old_value,
                    "simulated_value": new_value,
                    "linked_interventions": " | ".join(
                        sorted(changed_by.get(feature, set()))
                    ),
                })

synthetic_profiles = pd.DataFrame(profile_rows)
synthetic_changes = pd.DataFrame(change_rows)

print("Synthetic profile rows:", len(synthetic_profiles))
print("Synthetic change rows:", len(synthetic_changes))

display(synthetic_profiles.head())
display(synthetic_changes.head(20))


Synthetic profile rows: 60
Synthetic change rows: 547


,case_id,test_position,scenario,intervention_given,intervention_types,baseline_risk_probability,registered_course_count,registered_credits,repeated_course_count_current,previous_failed_course_count,previous_withdrawn_course_count,previous_repeated_course_count,previous_term_gpa_ordinal,previous_cgpa_ordinal,attendance_rate,absence_count,enrolled_course_count,accessed_course_count,course_access_rate,total_course_event_clicks,active_days_numeric,active_days_rate,zero_activity_days,largest_inactivity_days,learning_material_events_ordinal,assessment_interaction_events_ordinal,programme_or_school,year_level,previous_academic_standing
0,TEST_CASE_0085,84,conservative,1,Attendance support | Moodle engagement support | Referral and coordinated review | Workload adjustment,1.000000,6,50.0,0,0,0,0,NaN,NaN,5.0,0,2,1,0.5,0.0,5,0.147059,29,29,1.0,0.0,Bachelor of Engineering,1,NaN
1,TEST_CASE_0085,84,moderate,1,Attendance support | Moodle engagement support | Referral and coordinated review | Workload adjustment,1.000000,5,42.0,0,0,0,0,NaN,NaN,10.0,0,1,1,1.0,0.0,7,0.205882,27,27,1.0,0.0,Bachelor of Engineering,1,NaN
2,TEST_CASE_0085,84,strong,1,Attendance support | Moodle engagement support | Referral and coordinated review | Workload adjustment,1.000000,4,33.0,0,0,0,0,NaN,NaN,15.0,0,1,1,1.0,0.0,9,0.264706,25,25,2.0,0.0,Bachelor of Engineering,1,NaN
3,TEST_CASE_0724,723,conservative,1,Attendance support | Moodle engagement support | Referral and coordinated review | Workload adjustment,0.996667,6,50.0,0,0,0,0,NaN,NaN,5.0,0,1,1,1.0,0.0,5,0.147059,29,29,1.0,0.0,Bachelor of Engineering,1,NaN
4,TEST_CASE_0724,723,moderate,1,Attendance support | Moodle engagement support | Referral and coordinated review | Workload adjustment,0.996667,5,42.0,0,0,0,0,NaN,NaN,10.0,0,1,1,1.0,0.0,7,0.205882,27,27,1.0,0.0,Bachelor of Engineering,1,NaN


,case_id,test_position,scenario,feature,baseline_value,simulated_value,linked_interventions
0,TEST_CASE_0085,84,conservative,attendance_rate,0.0,5.000000,Attendance support
1,TEST_CASE_0085,84,conservative,accessed_course_count,0.0,1.000000,Moodle engagement support
2,TEST_CASE_0085,84,conservative,course_access_rate,0.0,0.500000,
3,TEST_CASE_0085,84,conservative,active_days_numeric,3.0,5.000000,Moodle engagement support
4,TEST_CASE_0085,84,conservative,active_days_rate,0.0,0.147059,
5,TEST_CASE_0085,84,conservative,zero_activity_days,34.0,29.000000,
6,TEST_CASE_0085,84,conservative,largest_inactivity_days,34.0,29.000000,Moodle engagement support
7,TEST_CASE_0085,84,conservative,learning_material_events_ordinal,0.0,1.000000,Moodle engagement support
8,TEST_CASE_0085,84,moderate,registered_course_count,6.0,5.000000,Workload adjustment
9,TEST_CASE_0085,84,moderate,registered_credits,50.0,42.000000,Workload adjustment


## 10. Validate all synthetic profiles


In [10]:
validation_rows = []

for _, row in synthetic_profiles.iterrows():
    baseline = X_test_raw.iloc[int(row["test_position"])]

    checks = {
        "attendance_rate": 0 <= row["attendance_rate"] <= 100,
        "absence_count": row["absence_count"] >= 0,
        "course_access_rate": 0 <= row["course_access_rate"] <= 1,
        "accessed_not_above_enrolled": (
            row["accessed_course_count"] <= row["enrolled_course_count"]
        ),
        "active_days": 0 <= row["active_days_numeric"] <= OBSERVATION_DAYS,
        "active_days_rate": 0 <= row["active_days_rate"] <= 1,
        "zero_activity_days": 0 <= row["zero_activity_days"] <= OBSERVATION_DAYS,
        "active_plus_zero": (
            int(round(row["active_days_numeric"] + row["zero_activity_days"]))
            == OBSERVATION_DAYS
        ),
        "largest_inactivity": (
            0 <= row["largest_inactivity_days"] <= row["zero_activity_days"]
        ),
        "learning_ordinal": (
            0 <= row["learning_material_events_ordinal"] <= 4
        ),
        "assessment_ordinal": (
            0 <= row["assessment_interaction_events_ordinal"] <= 4
        ),
        "click_cap": row["total_course_event_clicks"] <= CLICK_CAP,
    }

    for feature in IMMUTABLE_FEATURES & set(X_test_raw.columns):
        checks[f"immutable_{feature}"] = (
            (pd.isna(row[feature]) and pd.isna(baseline[feature]))
            or row[feature] == baseline[feature]
        )

    failed = [name for name, passed in checks.items() if not bool(passed)]

    validation_rows.append({
        "case_id": row["case_id"],
        "scenario": row["scenario"],
        "all_constraints_passed": len(failed) == 0,
        "failed_checks": " | ".join(failed),
    })

validation_results = pd.DataFrame(validation_rows)

display(
    validation_results["all_constraints_passed"]
    .value_counts(dropna=False)
    .rename_axis("all_constraints_passed")
    .to_frame("count")
)

failed_profiles = validation_results[
    ~validation_results["all_constraints_passed"]
]

if not failed_profiles.empty:
    display(failed_profiles)
    raise ValueError("Some synthetic profiles failed validation.")

print("All synthetic profiles passed the agreed constraints.")


,count
all_constraints_passed,
True,60


All synthetic profiles passed the agreed constraints.


## 11. Review one intervention case


In [12]:
example_case = case_summary.iloc[0]["case_id"]

print("Example case:", example_case)

print("Assigned interventions:")
display(
    intervention_log[
        intervention_log["case_id"] == example_case
    ]
)

print("Synthetic feature changes:")
display(
    synthetic_changes[
        synthetic_changes["case_id"] == example_case
    ].sort_values(["scenario", "feature"])
)


Example case: TEST_CASE_0085
Assigned interventions:


,case_id,support_area,trigger_features,recommended_action,intervention_given,intervention_status,test_position,predicted_risk_probability
0,TEST_CASE_0085,Attendance support,"absence_count, attendance_rate","Review attendance barriers, agree an attendance improvement plan, and monitor attendance weekly.",1,Planned,84,1.0
1,TEST_CASE_0085,Moodle engagement support,"active_days_rate, largest_inactivity_days, total_course_event_clicks","Provide an engagement plan covering course access, learning materials, assessment activity, and regular Moodle participation.",1,Planned,84,1.0
2,TEST_CASE_0085,Workload adjustment,registered_course_count,Review the student's registered workload and repeated courses with the academic adviser.,1,Planned,84,1.0
3,TEST_CASE_0085,Referral and coordinated review,High predicted risk or multiple risk domains,Refer the case for coordinated review by the academic adviser and the relevant student support service.,1,Planned,84,1.0


Synthetic feature changes:


,case_id,test_position,scenario,feature,baseline_value,simulated_value,linked_interventions
1,TEST_CASE_0085,84,conservative,accessed_course_count,0.0,1.000000,Moodle engagement support
3,TEST_CASE_0085,84,conservative,active_days_numeric,3.0,5.000000,Moodle engagement support
4,TEST_CASE_0085,84,conservative,active_days_rate,0.0,0.147059,
0,TEST_CASE_0085,84,conservative,attendance_rate,0.0,5.000000,Attendance support
2,TEST_CASE_0085,84,conservative,course_access_rate,0.0,0.500000,
6,TEST_CASE_0085,84,conservative,largest_inactivity_days,34.0,29.000000,Moodle engagement support
7,TEST_CASE_0085,84,conservative,learning_material_events_ordinal,0.0,1.000000,Moodle engagement support
5,TEST_CASE_0085,84,conservative,zero_activity_days,34.0,29.000000,
12,TEST_CASE_0085,84,moderate,accessed_course_count,0.0,1.000000,Moodle engagement support
14,TEST_CASE_0085,84,moderate,active_days_numeric,3.0,7.000000,Moodle engagement support


## 12. Save outputs for reassessment


In [ ]:
intervention_log.to_csv(
    OUTPUT_DIR / "intervention_log.csv",
    index=False,
)

synthetic_profiles.to_pickle(
    OUTPUT_DIR / "synthetic_profiles.pkl"
)

synthetic_profiles.to_csv(
    OUTPUT_DIR / "synthetic_profiles.csv",
    index=False,
)

synthetic_changes.to_csv(
    OUTPUT_DIR / "synthetic_feature_changes.csv",
    index=False,
)

validation_results.to_csv(
    OUTPUT_DIR / "synthetic_constraint_validation.csv",
    index=False,
)

metadata = {
    "objective": "O3",
    "scenarios": SCENARIOS,
    "observation_days": OBSERVATION_DAYS,
    "click_cap_percentile": 0.99,
    "click_cap_value": CLICK_CAP,
    "minimum_registered_credits": MIN_REGISTERED_CREDITS,
    "mutable_features": sorted(MUTABLE_FEATURES),
    "immutable_features": sorted(IMMUTABLE_FEATURES),
    "case_count": int(synthetic_profiles["case_id"].nunique()),
    "profile_count": int(len(synthetic_profiles)),
    "all_constraints_passed": bool(
        validation_results["all_constraints_passed"].all()
    ),
    "random_state": RANDOM_STATE,
}

with open(
    OUTPUT_DIR / "synthetic_progression_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(metadata, file, indent=2)

print("Synthetic progression outputs saved to:")
print(OUTPUT_DIR)


## Notebook 05 Summary

This notebook implemented the agreed intervention aware synthetic progression rules. Three scenarios were generated for each intervention case. Only mutable features linked to the assigned intervention were changed, while historical and contextual fields remained fixed.

Related features were recalculated to maintain consistency, and all synthetic profiles were checked against the agreed constraints. The generated profiles are ready to be processed by the saved preprocessing pipeline and selected model in the reassessment notebook.
